# 09 — Interactive Three-Body ICNN Sandbox

This notebook consolidates the three standalone scripts into one interactive workflow:

1. validate the 3D Newtonian three-body reference solver,
2. generate perturbed figure-eight trajectory data,
3. train baseline and ICNN-projected dynamics models,
4. run Exp7-style rollout-augmented V-only training,
5. compare solver vs neural autoregressive rollouts,
6. show interactive 3D trajectories with Plotly, so the camera can be rotated/zoomed directly in the notebook,
7. benchmark standard solver vs neural N-step rollout speed.

The three-body problem is used here as a nonlinear visual stress test. The pure Newtonian system is conservative, while the ICNN projection enforces a Lyapunov decrease condition, so the projected model may damp or distort the orbit. This is useful for studying the method's limitations.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from IPython.display import display

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for p in [start, *start.parents]:
        if (p / "three_body").exists() and (p / "stable_icnn_physics").exists():
            return p
    raise RuntimeError("Start Jupyter from the DeepLearningFTN repository.")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

from stable_icnn_physics.eval import autoregressive_rollout_model
from stable_icnn_physics.rollout_augmented import projection_diagnostics
from stable_icnn_physics.train import evaluate_derivative_mse
from three_body import ThreeBodyConfig, ThreeBodySystem3D, figure_eight_state_3d
from three_body.data import (
    dataset_path,
    generate_reference_figure_eight,
    generate_trajectory_dataset,
    load_trajectory_dataset,
    save_trajectory_dataset,
    trajectory_pairs_to_derivatives,
)
from three_body.eval import benchmark_n_step, rmse_per_step, summarize_rollout
from three_body.models import build_baseline_model, build_stable_icnn_model
from three_body.system import integrate_rk4_fixed
from three_body.train import (
    make_derivative_dataset_from_trajectories,
    train_exp7_style_v_only,
    train_phase1_models,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

DATA_DIR = REPO_ROOT / "data" / "three_body"
CKPT_DIR = REPO_ROOT / "checkpoints" / "three_body"
RESULTS_DIR = REPO_ROOT / "results" / "three_body"
PLOTS_DIR = RESULTS_DIR / "plots"
for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_ROOT)
print("device:", DEVICE)
print("torch:", torch.__version__)

## Configuration

Use `QUICK=True` for a smoke test. Set `QUICK=False` for a fuller run. The notebook reuses saved datasets/checkpoints unless `FORCE_RETRAIN=True`.

In [ ]:
QUICK = True
FORCE_RETRAIN = False

SEED = 0
TAG = "three_body_figure8_icnn_quick" if QUICK else "three_body_figure8_icnn"

DT = 0.005
STEPS = 200 if QUICK else 700
TRAIN_TRAJS = 16 if QUICK else 96
VAL_TRAJS = 4 if QUICK else 16
TEST_TRAJS = 2 if QUICK else 8
PHASE1_EPOCHS = 20 if QUICK else 150
V_ONLY_EPOCHS = 20 if QUICK else 250
BATCH_SIZE = 256 if QUICK else 512
ROLLOUT_TRAIN = 16 if QUICK else 128
ROLLOUT_VAL = 4 if QUICK else 32

HIDDEN = 256
DEPTH = 3
LYAPUNOV_HIDDEN = 128
ALPHA = 1e-6

cfg = ThreeBodyConfig(
    gravity=1.0,
    masses=(1.0, 1.0, 1.0),
    softening=1e-3,
    damping=0.0,
    sample_noise_pos=0.02,
    sample_noise_vel=0.02,
    sample_z_noise=0.0,
)
system = ThreeBodySystem3D(cfg)

print(system.metadata())
print({"TAG": TAG, "STEPS": STEPS, "TRAIN_TRAJS": TRAIN_TRAJS, "PHASE1_EPOCHS": PHASE1_EPOCHS, "V_ONLY_EPOCHS": V_ONLY_EPOCHS})

## Interactive Plotly helpers

These figures are interactive: drag to rotate, scroll to zoom, double-click to reset the camera.

In [ ]:
def _positions(traj: np.ndarray) -> np.ndarray:
    arr = np.asarray(traj, dtype=np.float32)
    if arr.shape[-1] != 18:
        raise ValueError("last dimension must be 18")
    return arr[..., :9].reshape(*arr.shape[:-1], 3, 3)

def plotly_3d_paths(trajectories: dict[str, np.ndarray], title: str = "3D trajectories", stride: int = 1) -> go.Figure:
    fig = go.Figure()
    for label, traj in trajectories.items():
        pos = _positions(traj)[::stride]
        for body in range(3):
            fig.add_trace(go.Scatter3d(
                x=pos[:, body, 0], y=pos[:, body, 1], z=pos[:, body, 2],
                mode="lines",
                name=f"{label} body {body+1}",
                line=dict(width=4 if label == "solver" else 2),
            ))
            fig.add_trace(go.Scatter3d(
                x=[pos[0, body, 0]], y=[pos[0, body, 1]], z=[pos[0, body, 2]],
                mode="markers",
                name=f"{label} start {body+1}",
                marker=dict(size=4, symbol="circle"),
                showlegend=False,
            ))
            fig.add_trace(go.Scatter3d(
                x=[pos[-1, body, 0]], y=[pos[-1, body, 1]], z=[pos[-1, body, 2]],
                mode="markers",
                name=f"{label} end {body+1}",
                marker=dict(size=5, symbol="x"),
                showlegend=False,
            ))
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x", yaxis_title="y", zaxis_title="z",
            aspectmode="data",
        ),
        legend=dict(itemsizing="constant"),
        height=700,
    )
    return fig

def plotly_3d_animation(trajectories: dict[str, np.ndarray], title: str = "3D rollout animation", stride: int = 2, trail: int = 80) -> go.Figure:
    labels = list(trajectories.keys())
    pos_dict = {label: _positions(traj)[::stride] for label, traj in trajectories.items()}
    n_frames = min(pos.shape[0] for pos in pos_dict.values())
    trail = max(1, int(trail))

    def frame_traces(k: int):
        traces = []
        start = max(0, k - trail)
        for label in labels:
            pos = pos_dict[label]
            for body in range(3):
                traces.append(go.Scatter3d(
                    x=pos[start:k+1, body, 0],
                    y=pos[start:k+1, body, 1],
                    z=pos[start:k+1, body, 2],
                    mode="lines",
                    name=f"{label} trail {body+1}",
                    line=dict(width=4 if label == "solver" else 2),
                    showlegend=(k == 0),
                ))
                traces.append(go.Scatter3d(
                    x=[pos[k, body, 0]],
                    y=[pos[k, body, 1]],
                    z=[pos[k, body, 2]],
                    mode="markers",
                    name=f"{label} body {body+1}",
                    marker=dict(size=6 if label == "solver" else 4),
                    showlegend=(k == 0),
                ))
        return traces

    fig = go.Figure(data=frame_traces(0))
    fig.frames = [go.Frame(data=frame_traces(k), name=str(k)) for k in range(n_frames)]

    steps = [
        dict(
            method="animate",
            args=[[str(k)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
            label=str(k),
        )
        for k in range(0, n_frames, max(1, n_frames // 20))
    ]
    fig.update_layout(
        title=title,
        height=750,
        scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="data"),
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            buttons=[
                dict(label="Play", method="animate", args=[None, {"frame": {"duration": 40, "redraw": True}, "fromcurrent": True}]),
                dict(label="Pause", method="animate", args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
            ],
        )],
        sliders=[dict(active=0, steps=steps, x=0.05, len=0.9)],
    )
    return fig

def save_plotly(fig: go.Figure, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(path, include_plotlyjs="cdn")
    print("saved:", path)

## 1. Reference solver: solve_ivp vs RK4

This replaces `scripts/run_three_body_reference.py` inside the notebook.

In [ ]:
x0_ref = figure_eight_state_3d()
REF_STEPS = 400 if QUICK else 1400

print("generating solve_ivp reference...")
ref_traj_batch, ref_t = generate_reference_figure_eight(system, x0_ref, steps=REF_STEPS, dt=DT, solver="solve_ivp")
ref_traj = ref_traj_batch[0]

print("generating RK4 reference...")
rk4_ref_traj = integrate_rk4_fixed(system, x0_ref, steps=REF_STEPS, dt=DT)
err_rk4 = rmse_per_step(system, ref_traj, rk4_ref_traj)

reference_summary = {
    "steps": REF_STEPS,
    "dt": DT,
    "rk4_final_rmse_vs_solve_ivp": float(err_rk4[-1]),
    "rk4_mean_rmse_vs_solve_ivp": float(err_rk4.mean()),
    "solve_ivp_energy_start": float(system.energy(ref_traj[[0]])[0]),
    "solve_ivp_energy_end": float(system.energy(ref_traj[[-1]])[0]),
    "rk4_energy_start": float(system.energy(rk4_ref_traj[[0]])[0]),
    "rk4_energy_end": float(system.energy(rk4_ref_traj[[-1]])[0]),
}
print(json.dumps(reference_summary, indent=2))

out_npz = DATA_DIR / "figure8_reference_solve_ivp.npz"
save_trajectory_dataset(out_npz, ref_traj_batch, ref_t, metadata={**reference_summary, "solver": "solve_ivp_DOP853"})
print("saved:", out_npz)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ref_t, err_rk4)
plt.xlabel("time")
plt.ylabel("RMSE vs solve_ivp")
plt.title("Fixed RK4 vs DOP853 reference")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig = plotly_3d_paths({"solver": ref_traj, "rk4": rk4_ref_traj}, title="Interactive figure-eight: solve_ivp vs RK4")
save_plotly(fig, RESULTS_DIR / "reference" / "figure8_solve_ivp_vs_rk4_interactive.html")
fig.show()

### Interactive reference animation

This animation remains rotatable while playing/paused. Use the Plotly toolbar/camera controls.

In [ ]:
fig_anim_ref = plotly_3d_animation(
    {"solver": ref_traj, "rk4": rk4_ref_traj},
    title="Interactive animation: solve_ivp vs RK4",
    stride=2 if QUICK else 5,
    trail=80,
)
save_plotly(fig_anim_ref, RESULTS_DIR / "reference" / "figure8_solve_ivp_vs_rk4_animation.html")
fig_anim_ref.show()

## 2. Generate/load training trajectories

This replaces dataset creation inside `scripts/run_three_body_training.py`.

In [ ]:
def load_or_generate_dataset(path: Path, n_trajs: int, split: str):
    if path.exists():
        traj, t, meta = load_trajectory_dataset(path)
        print("loaded", path)
        return traj, t, meta
    traj, t = generate_trajectory_dataset(system, n_trajs, STEPS, DT, split=split, seed=SEED, solver="rk4")
    meta = {"system": system.metadata(), "split": split, "dt": DT, "steps": STEPS, "solver": "rk4"}
    save_trajectory_dataset(path, traj, t, meta)
    print("saved", path)
    return traj, t, meta

train_path = dataset_path(DATA_DIR, "train_perturbed", TRAIN_TRAJS, STEPS, DT, SEED, "rk4")
val_path = dataset_path(DATA_DIR, "val_perturbed", VAL_TRAJS, STEPS, DT, SEED, "rk4")
test_path = dataset_path(DATA_DIR, "test_perturbed", TEST_TRAJS, STEPS, DT, SEED, "rk4")

train_traj, t, _ = load_or_generate_dataset(train_path, TRAIN_TRAJS, "train")
val_traj, _, _ = load_or_generate_dataset(val_path, VAL_TRAJS, "val")
test_traj, _, _ = load_or_generate_dataset(test_path, TEST_TRAJS, "test")

train_ds = make_derivative_dataset_from_trajectories(train_traj, DT)
val_ds = make_derivative_dataset_from_trajectories(val_traj, DT)
test_ds = make_derivative_dataset_from_trajectories(test_traj, DT)
x_train_states, _ = trajectory_pairs_to_derivatives(train_traj, DT)
x_val_states, _ = trajectory_pairs_to_derivatives(val_traj, DT)

print("train_traj:", train_traj.shape)
print("val_traj:", val_traj.shape)
print("test_traj:", test_traj.shape)
print("derivative train pairs:", len(train_ds))

## 3. Train/load baseline and ICNN models

This replaces the model training part of `scripts/run_three_body_training.py`.

In [ ]:
stable_ckpt = CKPT_DIR / f"{TAG}_phase1_stable.pt"
baseline_ckpt = CKPT_DIR / f"{TAG}_baseline.pt"
exp7_ckpt = CKPT_DIR / f"{TAG}_exp7style_stable.pt"

if FORCE_RETRAIN or not (stable_ckpt.exists() and baseline_ckpt.exists()):
    stable_phase1, baseline = train_phase1_models(
        train_ds,
        val_ds,
        stable_ckpt,
        baseline_ckpt,
        hidden=HIDDEN,
        depth=DEPTH,
        lyapunov_hidden=LYAPUNOV_HIDDEN,
        alpha=ALPHA,
        epochs=PHASE1_EPOCHS,
        batch_size=BATCH_SIZE,
        device=DEVICE,
    )
else:
    stable_phase1 = build_stable_icnn_model(hidden=HIDDEN, depth=DEPTH, lyapunov_hidden=LYAPUNOV_HIDDEN, alpha=ALPHA)
    baseline = build_baseline_model(hidden=HIDDEN, depth=DEPTH)
    stable_phase1.load_state_dict(torch.load(stable_ckpt, map_location=DEVICE, weights_only=True)["model_state"])
    baseline.load_state_dict(torch.load(baseline_ckpt, map_location=DEVICE, weights_only=True)["model_state"])
    stable_phase1.to(DEVICE).eval()
    baseline.to(DEVICE).eval()
    print("loaded phase-1 checkpoints")

print("stable_ckpt:", stable_ckpt)
print("baseline_ckpt:", baseline_ckpt)

## 4. Exp7-style V-only rollout-augmented training

This freezes the newly trained `fhat`, generates model-induced rollout states, and trains only `V`.

In [ ]:
if FORCE_RETRAIN or not exp7_ckpt.exists():
    stable_exp7 = build_stable_icnn_model(hidden=HIDDEN, depth=DEPTH, lyapunov_hidden=LYAPUNOV_HIDDEN, alpha=ALPHA)
    stable_exp7.load_state_dict(torch.load(stable_ckpt, map_location=DEVICE, weights_only=True)["model_state"])
    stable_exp7.to(DEVICE).eval()
    stable_exp7, history_v, rollout_states = train_exp7_style_v_only(
        stable_exp7,
        system,
        x_train_states,
        x_val_states,
        DT,
        STEPS,
        n_rollout_train=ROLLOUT_TRAIN,
        n_rollout_val=ROLLOUT_VAL,
        epochs=V_ONLY_EPOCHS,
        batch_size=BATCH_SIZE,
        device=DEVICE,
    )
    torch.save({"model_state": stable_exp7.state_dict(), "history": history_v.__dict__}, exp7_ckpt)
    print("saved:", exp7_ckpt)
else:
    stable_exp7 = build_stable_icnn_model(hidden=HIDDEN, depth=DEPTH, lyapunov_hidden=LYAPUNOV_HIDDEN, alpha=ALPHA)
    stable_exp7.load_state_dict(torch.load(exp7_ckpt, map_location=DEVICE, weights_only=True)["model_state"])
    stable_exp7.to(DEVICE).eval()
    history_v = None
    print("loaded Exp7-style checkpoint")

print("exp7_ckpt:", exp7_ckpt)

In [ ]:
if history_v is not None:
    plt.figure(figsize=(8, 4))
    plt.semilogy(history_v.epoch, history_v.train_loss, label="train")
    plt.semilogy(history_v.epoch, history_v.val_loss, label="val")
    plt.semilogy(history_v.epoch, history_v.best_val_loss, label="best val", linestyle="--")
    plt.xlabel("epoch")
    plt.ylabel("V-only violation loss")
    plt.title("Three-body Exp7-style V-only training")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("V-only history not available because checkpoint was loaded.")

## 5. Autoregressive rollout evaluation

All neural models are rolled autoregressively from their own previous predictions.

In [ ]:
true_traj = test_traj[0]
x0 = true_traj[0:1]

baseline_traj = autoregressive_rollout_model(baseline, x0, STEPS, DT, device=DEVICE)[:, 0]
fhat_traj = autoregressive_rollout_model(stable_exp7.fhat, x0, STEPS, DT, device=DEVICE)[:, 0]
phase1_traj = autoregressive_rollout_model(stable_phase1, x0, STEPS, DT, device=DEVICE)[:, 0]
exp7_traj = autoregressive_rollout_model(stable_exp7, x0, STEPS, DT, device=DEVICE)[:, 0]

summary = {
    "experiment": TAG,
    "quick": QUICK,
    "system": system.metadata(),
    "dt": DT,
    "steps": STEPS,
    "train_trajs": TRAIN_TRAJS,
    "val_trajs": VAL_TRAJS,
    "test_trajs": TEST_TRAJS,
    "derivative_mse": {
        "baseline": float(evaluate_derivative_mse(baseline, test_ds, device=DEVICE)),
        "fhat": float(evaluate_derivative_mse(stable_exp7.fhat, test_ds, device=DEVICE)),
        "phase1_stable": float(evaluate_derivative_mse(stable_phase1, test_ds, device=DEVICE)),
        "exp7_stable": float(evaluate_derivative_mse(stable_exp7, test_ds, device=DEVICE)),
    },
    "rollout": {
        "baseline": summarize_rollout(system, true_traj, baseline_traj).as_dict(),
        "fhat": summarize_rollout(system, true_traj, fhat_traj).as_dict(),
        "phase1_stable": summarize_rollout(system, true_traj, phase1_traj).as_dict(),
        "exp7_stable": summarize_rollout(system, true_traj, exp7_traj).as_dict(),
    },
    "projection_exp7": {
        k: v for k, v in projection_diagnostics(stable_exp7, exp7_traj[:, None], device=DEVICE).items()
        if k not in {"fire", "correction_norm", "violation", "V", "dV"}
    },
}

summary_path = RESULTS_DIR / f"{TAG}_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("saved:", summary_path)

In [ ]:
err_curves = {
    "baseline": rmse_per_step(system, true_traj, baseline_traj),
    "fhat only": rmse_per_step(system, true_traj, fhat_traj),
    "phase1 stable": rmse_per_step(system, true_traj, phase1_traj),
    "Exp7-style stable": rmse_per_step(system, true_traj, exp7_traj),
}

plt.figure(figsize=(9, 5))
for name, err in err_curves.items():
    plt.plot(t, err, label=name)
plt.xlabel("time")
plt.ylabel("RMSE")
plt.title("Three-body autoregressive rollout error")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOTS_DIR / f"{TAG}_rollout_rmse.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

display(pd.DataFrame(summary["rollout"]).T)

## 6. Interactive 3D comparison

Rotate, pan, and zoom the 3D plot directly in the notebook.

In [ ]:
fig_paths = plotly_3d_paths(
    {
        "solver": true_traj,
        "baseline": baseline_traj,
        "fhat": fhat_traj,
        "icnn": exp7_traj,
    },
    title="Interactive 3D paths: solver vs neural rollouts",
    stride=1 if QUICK else 2,
)
save_plotly(fig_paths, RESULTS_DIR / f"{TAG}_interactive_paths.html")
fig_paths.show()

### Interactive animated comparison

This can be heavier than the static interactive path plot. For a full run, increase `stride` if the notebook becomes slow.

In [ ]:
fig_anim = plotly_3d_animation(
    {
        "solver": true_traj,
        "icnn": exp7_traj,
    },
    title="Interactive animation: solver vs ICNN projected rollout",
    stride=2 if QUICK else 5,
    trail=80,
)
save_plotly(fig_anim, RESULTS_DIR / f"{TAG}_interactive_animation_solver_vs_icnn.html")
fig_anim.show()

## 7. Real-time / N-step benchmark

This replaces `scripts/run_three_body_benchmark.py` inside the notebook. It compares standard solvers with neural autoregressive rollouts.

In [ ]:
BENCH_STEPS = 200 if QUICK else 1000
bench_result = benchmark_n_step(
    system,
    figure_eight_state_3d(),
    steps=BENCH_STEPS,
    dt=DT,
    models={
        "baseline_nn": baseline,
        "fhat_only": stable_exp7.fhat,
        "icnn_projected": stable_exp7,
    },
    device=DEVICE,
)
bench_result["metadata"] = {"tag": TAG, "steps": BENCH_STEPS, "dt": DT, "device": str(DEVICE), "system": system.metadata()}

bench_df = pd.DataFrame({k: v for k, v in bench_result.items() if isinstance(v, dict) and k != "metadata"}).T
display(bench_df)

out_json = RESULTS_DIR / f"{TAG}_realtime_benchmark_steps{BENCH_STEPS}.json"
out_csv = RESULTS_DIR / f"{TAG}_realtime_benchmark_steps{BENCH_STEPS}.csv"
out_json.write_text(json.dumps(bench_result, indent=2), encoding="utf-8")
bench_df.to_csv(out_csv)
print("saved:", out_json)
print("saved:", out_csv)